In [1]:
# Install necessary packages
%pip install matplotlib networkx pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 14.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pyvis]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import networkx as nx
import os
# from yafs.topology import Topology # Not needed when loading GEXF directly

In [3]:
topology_name = "cloud1-gateway12-fog12-end48"
# Load the topology from the GEXF file
# Make sure to run create_topology.py first to generate this file
gexf_path = f"topologies/{topology_name}.gexf"

if os.path.exists(gexf_path):
    G = nx.read_gexf(gexf_path)
    print(f"Successfully loaded graph from {gexf_path}")
    print(f"Nodes: {len(G.nodes)}")
    print(f"Edges: {len(G.edges)}")
else:
    print(f"File not found: {gexf_path}")
    print("Please run 'python create_topology.py' to generate the topology file.")
    # Create an empty graph to prevent errors in next cell
    G = nx.Graph()

Successfully loaded graph from topologies/cloud1-gateway12-fog12-end48.gexf
Nodes: 74
Edges: 73


In [4]:
from pyvis.network import Network
import os
import webbrowser

# Create a PyVis network
net = Network(notebook=True, height="750px", width="100%", cdn_resources='in_line')


def level_for_model(model_value: str) -> int:
    v = (model_value or "").lower()
    if "cloud" in v:
        return 0
    if "proxy" in v:
        return 1
    if "gateway" in v:
        return 2
    if "fog" in v:
        return 3
    # sensors/ends/actuators go to the bottom layer
    return 4


# Add nodes with colors and hierarchical level
for n in G.nodes():
    model = G.nodes[n].get('model', 'Unknown')
    model_lower = model.lower()
    label = G.nodes[n].get('label', str(n))

    color = '#808080'
    if 'cloud' in model_lower:
        color = '#FF0000'
    elif 'proxy' in model_lower:
        color = '#00AA00'
    elif 'gateway' in model_lower:
        color = '#00CED1'  # teal for gateways
    elif 'fog' in model_lower:
        color = '#FFA500'
    elif 'sensor' in model_lower or 'end' in model_lower or 'device' in model_lower:
        color = '#800080'

    level = level_for_model(model)
    title = f"ID: {n}\nType: {model}"
    net.add_node(n, label=label, title=title, color=color, level=level)

# Add edges
for s, d in G.edges():
    net.add_edge(s, d)

# Force hierarchical top-to-bottom layout
net.set_options("""{
  "layout": {"hierarchical": {"enabled": true, "direction": "UD", "sortMethod": "hubsize"}},
  "physics": {"enabled": false}
}""")

# Display the graph
html_content = net.generate_html()
output_path = f"topologies/{topology_name}.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_content)
webbrowser.open('file://' + os.path.abspath(output_path))

True